# Marbl's Model Diagnostics 
Authors: Aleksandar Gradev, Aleksandar Milosavljevic, Pavle Cvijanovic, Vincent Skakala

...

In [6]:
# Load libraries
from src.utils       import (
    get_project_root, load_secrets, 
    add_dates_to_output, ZONE_SHORT_CODE
)
from src.evaluation  import avg_errors
from src.diagnostics import (
    get_intraday_volatility, plot_mean_volatility,
    animate_volatility_timeseries, plot_error_vs_volatility,
    evaluate_cluster_performance
)
import pandas as pd
# Load data
# should load proba_dfs, clusters_dfs, pred_dfs from the pipeline_summary

In [7]:
ROOT          = get_project_root()
RAW_DIR       = ROOT / 'data' / 'raw'
CLEAN_DIR     = ROOT / 'data' / 'clean'
PROCESSED_DIR = ROOT / 'data' / 'processed'
MODELS_DIR    = ROOT / 'data' / 'models'
LIVE_DIR      = ROOT / 'data' / 'live'

MODELS_DIR.mkdir(parents=True, exist_ok=True)

# ----------------- Zone configuration -----------------
# REMOVE EVERYTHING WE DONT NEED FOR THE DIAGNOSTICS !
ZONES = {
    'DK1': {
        'entsoe_code': '10YDK-1--------W',
        'k_focus':     6,
        'best_p':      20,
        'era5_years':  [2023, 2024, 2025],
        # 'era5_coords': ERA5_ZONE_COORDS['DK1'],
        'cities':      ['Copenhagen', 'Aarhus'],
        'eval_cutoff': pd.Timestamp('2025-06-18'),
    },
    'ES': {
        'entsoe_code': '10YES-REE------0',
        'k_focus':     3,
        'best_p':      20,
        'era5_years':  [2023, 2024, 2025],
        # 'era5_coords': ERA5_ZONE_COORDS['ES'],
        'cities':      ['Madrid', 'Barcelona'],
        'eval_cutoff': pd.Timestamp('2025-06-20'),
    },
    'NO2': {
        'entsoe_code': '10YNO-2--------T',
        'k_focus':     5,
        'best_p':      7,
        'era5_years':  [2023, 2024, 2025],
        # 'era5_coords': ERA5_ZONE_COORDS['NO2'],
        'cities':      ['Kristiansand', 'Stavanger'],
        'eval_cutoff': pd.Timestamp('2025-06-20'),
    },
}

print(f'Project root : {ROOT}')
print(f'Zones        : {list(ZONES)}')
print(f'Short codes  : {ZONE_SHORT_CODE}')


Project root : C:\WU\SBWL Data Science\Course 5 - Data Science Lab\Old project\DsLab25W_marbl.energy\summary
Zones        : ['DK1', 'ES', 'NO2']
Short codes  : {'DK1': 'DK', 'ES': 'ES', 'NO2': 'NO'}


In [8]:
# Load Mastersets
masterset_dfs = {}

for zone in ZONES:
    path = PROCESSED_DIR / f'{zone}_masterset.csv'
    if not path.exists():
        print(f'WARNING: {zone}_masterset.csv not found — downstream steps will skip {zone}.')
        continue
    df = pd.read_csv(path)
    df = df.rename(columns={df.columns[0]: 'date_time'})
    masterset_dfs[zone] = df

print(f'\nLoaded mastersets: {list(masterset_dfs)}')



Loaded mastersets: ['DK1', 'ES', 'NO2']


## Intraday Volatility Diagnostic

With this diagnostic we aim to examine the intraday dynamics in the researched markets and understand in more details where do the models fail.

We start by looking if the models can reproduce the intraday volatility structure of actual electricity prices. Intraday volatility is computed for each day as the variation in hourly prices within that day, capturing how turbulent or stable the market was. First, we will look at the timeseries of the daily intraday volatility of both actual and predicted prices over the full evaluation (testing) period. We want to check whether the model systematically underestimates or overestimates turbulence and whether it can capture high-volatility events. Second, we will compare the mean intraday volatility of the actual price against the mean volatility produced by each individual predicted cluster, exposing whether the cluster architecture itself introduces a structural smoothing bias.

After that we will investigate the relationship between market turbulence and forecast error quality. For each day in the test set, actual intraday volatility is plotted against the forecast MAE for that day in a scatterplot. Both axes share the same unit (EUR/MWh), making the relationship directly interpretable. The purpose is to test whether the model's forecast errors are systematically driven by high-volatility days or whether errors are distributed uniformly regardless of market conditions. A positive relationship would confirm that the model's performance degrades specifically under turbulent market conditions, linking the structural findings of Diagnostic 2 directly to measurable forecasting consequences.

In [ ]:
# Select zone
zone = "DK1"

# Define path to the save folder (remove this and use default False parameter
# in the function for the final notebook, after we have all videos saved)
path_to_save = r"C:\WU\SBWL Data Science\Course 5 - Data Science Lab\Midterm Presentation\intraday_vol_"

# Extract fundamental datasets for the chosen zone
df_m  = masterset_dfs[zone]
df_p  = pred_dfs[zone]
df_pr = proba_dfs[zone]
df_c  = clusters_dfs[zone]

# Plot the Mean Volatilities Bar Chart
plot_mean_volatility(df_m, df_p, df_pr, df_c)

# Render the Timeseries Animation
animate_volatility_timeseries(df_m, df_p, df_pr, df_c, save = True, 
                              path_and_name=path_to_save + zone + ".mp4")

# Plot the Scatter Plot 
plot_error_vs_volatility(df_m, df_p, df_pr, df_c, avg_errors)

Some comments here...

## Layer 1 vs Layer 2 Error Diagnostic 

The current MARBL forecasting model consists of Layer 1 (regime classifier that gives cluster probabilities) and Layer 2 (separate forecaster that generates hourly price forcasts per cluster).

This diagnostic isolates which component is causing forecast errors. We split test results into two groups: days where Layer 1 correctly predicted the cluster (assigned the highest probability to the correct cluster) versus days where it was wrong. By comparing forecast accuracy (WAPE, Mean Absolute Error) between these groups, we identify whether Layer 1's regime misclassification or Layer 2's forecasting within clusters is the primary bottleneck.


In [ ]:
for zone, cfg in ZONES.items():
    evaluate_cluster_performance(zone, proba_dfs, clusters_dfs, pred_dfs)

### Top-1 vs Top-3 Accuracy

The gap between Top-1 and Top-3 accuracy is the first key finding. In all three zones, the correct cluster falls within the classifier's top 3 predictions almost perfectly. However, top-1 accuracy is considerably lower. This means the classifier consistently identifies the right neighbourhood of regimes but struggles to commit to the correct one. 

We believe this is a direct consequence of the overlapping cluster centroids. Ward linkage produces clusters that are not well-separated, so the decision boundary between adjacent regimes is genuinely ambiguous. Our proposed Gaussian HMM addresses this by design, as its probabilistic regime assignments naturally handle boundary uncertainty.

### WAPE Gap Between Correct and Incorrect Cluster Days

The WAPE difference between days where Layer 1 was correct and days where it was incorrect varies substantially across zones, and this variation is the most actionable finding of this diagnostic.

In **NO2**, the gap is 12.6 percentage points (19.1% vs 31.7%). This is by far the largest gap across all three zones, meaning that Layer 1 misclassification is disproportionately costly here. When the classifier assigns the wrong regime, the Layer 2 regression model produces predictions that are  structurally misaligned with the actual price profile. Fixing Layer 1 through the HMM transition is therefore the highest-priority intervention for NO2.

In **DK1** and **ES**, the gap is narrow, 2.6pp and 2.3pp respectively. Even when Layer 1 assigns the wrong cluster, the WAPE barely changes. This tells us that in these two zones, the clusters are similar enough in their price profiles that a wrong assignment does not dramatically worsen the regression output. The error is not coming from regime misclassification, but from within the Layer 2 regression models themselves, regardless of which cluster is assigned. The bottleneck in DK1 and ES is therefore Layer 2 feature poverty, i.e. the absence of gas prices, load forecasts and zone-specific supply variables that the regression models need to produce accurate hourly predictions within any regime.
